In [14]:
# Prediction table (tab:prediction_models): fit update rules on train questions
# x 8 signed random graphs, score test questions by the paper's flip-and-class
# balanced accuracy (Appendix C.5) and raw accuracy; seen (In-d.) vs fresh (OOD).
import json
import numpy as np
import pandas as pd

from utils import (RES_DIR, MODELS, S_PREV, FIELD, D_POS, D_NEG, Logistic, load_data,
                   build_xy, two_stage_fit, discrete_rollout, fcba, raw_acc)

runs, banks, P, GCLASS = load_data()
REGIMES = ["subjective", "objective"]
T = 8                            # transitions per episode: s(0..8)
print(len(runs), "episodes")

9600 episodes


In [15]:
# x (E, 8, 32, 19) = [s_prev | bias p q p⊗q | drive_pos drive_neg] per
# (transition, agent); y (E, 8, 32) = the next spin s(t+1) (0 = unparsed)
x, y, ep = build_xy("gpt-4o-mini", "objective", runs, banks, P, GCLASS)
print("x", x.shape, " y", y.shape)

x (1600, 8, 32, 19)  y (1600, 8, 32)


In [16]:
# the table methods; each returns (onestep, rollout) test predictions
def predict_all(x_tr, y_tr, x_te, J_te):
    rows = x_tr.reshape(-1, x_tr.shape[-1])          # pooled train transitions
    parsed = y_tr.reshape(-1) != 0                   # unparsed targets excluded
    y01 = (y_tr.reshape(-1)[parsed] > 0).astype(float)
    E, s0, phi = len(x_te), x_te[:, 0, :, S_PREV], x_te[:, 0, :, FIELD]
    out = {}

    # majority class: the train-majority spin, everywhere (raw table only;
    # it scores exactly 50 under the flip-and-class balanced metric)
    c = 1 if y01.mean() >= 0.5 else -1
    const = np.full((E, T, 32), c, dtype=int)
    out["Majority Class"] = (const, const)

    # persistence: no change; rollout frozen at s(0)
    out["Persistence"] = (x_te[..., S_PREV].astype(int),
                          np.repeat(s0[:, None].astype(int), T, axis=1))

    # interaction-free: logistic on the static field only (state-independent)
    clf = Logistic().fit(rows[parsed][:, FIELD], y01)
    pred = clf.predict_spin(x_te[..., FIELD])
    out["Interaction-Free"] = (pred, pred)

    # mean-field (Curie-Weiss): [field | population mean s̄(t)]
    def with_sbar(xx):
        sbar = xx[..., S_PREV].mean(axis=-1)
        return np.concatenate([xx[..., FIELD],
                               np.broadcast_to(sbar[..., None, None],
                                               xx.shape[:-1] + (1,))], axis=-1)
    clf = Logistic().fit(with_sbar(x_tr).reshape(-1, 17)[parsed], y01)
    onestep = clf.predict_spin(with_sbar(x_te))
    s, rollout = s0.copy(), np.empty((E, T, 32), dtype=int)
    for t in range(T):
        sbar = np.broadcast_to(s.mean(axis=1)[:, None, None], (E, 32, 1))
        s = clf.predict_spin(np.concatenate([phi, sbar], axis=-1)).astype(float)
        rollout[:, t] = s
    out["Mean-Field (Curie-Weiss)"] = (onestep, rollout)

    # discrete update with 1 coupling [(J s)]
    pos, neg = rows[:, D_POS], rows[:, D_NEG]
    clf = Logistic().fit(np.concatenate(
        [rows[:, FIELD], (pos + neg)[:, None]], axis=-1)[parsed], y01)
    dr = (x_te[..., D_POS] + x_te[..., D_NEG])[..., None]
    onestep = clf.predict_spin(np.concatenate([x_te[..., FIELD], dr], axis=-1))
    out["Discrete Update"] = (onestep, discrete_rollout(phi, J_te, s0, clf.w, 1))

    # + 3 couplings [(J+ s) | (J- s) | (|J| s)]: collinear, so fit in two stages
    w = two_stage_fit(rows[:, FIELD], np.stack([pos, neg], axis=-1),
                      pos - neg, parsed, y01)
    dr = np.stack([x_te[..., D_POS], x_te[..., D_NEG],
                   x_te[..., D_POS] - x_te[..., D_NEG]], axis=-1)
    onestep = np.where(np.concatenate([x_te[..., FIELD], dr], axis=-1) @ w > 0, 1, -1)
    out["+ 3 Couplings"] = (onestep, discrete_rollout(phi, J_te, s0, w, 3))
    return out

In [17]:
# metrics from utils: fcba is the paper's flip-and-class balanced accuracy
# (Appendix C.5), the unweighted mean over (flip vs stay) x (next spin +-1);
# raw_acc is plain accuracy. Majority Class only appears under raw accuracy.
METRIC_ROWS = {
    "balanced": ["Persistence", "Interaction-Free", "Mean-Field (Curie-Weiss)",
                 "Discrete Update", "+ 3 Couplings"],
    "raw": ["Majority Class", "Persistence", "Interaction-Free",
            "Mean-Field (Curie-Weiss)", "Discrete Update", "+ 3 Couplings"]}
METRIC_NAMES = {"balanced": "flip-and-class balanced accuracy",
                "raw": "raw accuracy"}

In [18]:
# fit on train questions x J0-J7 (lattices excluded), score on test
# questions x seen (in-d.) vs fresh (out-d.) random graphs, both metrics
SPLITS = [("In-d.", "seen"), ("Out-d.", "fresh")]
results = {}
for regime in REGIMES:
    for model, label, short in MODELS:
        x, y, ep = build_xy(model, regime, runs, banks, P, GCLASS)
        train = (ep["split"] == "train") & np.isin(ep["gcls"], ["seen", "train_only"])
        test = (ep["split"] == "test") & np.isin(ep["gcls"], ["seen", "fresh"])
        preds = predict_all(x[train], y[train], x[test], ep["J"][test])
        y_te, s0, gc = y[test], x[test][:, 0, :, S_PREV], ep["gcls"][test]
        score = {"balanced": lambda p, m: fcba(p, y_te, s0, m),
                 "raw": lambda p, m: raw_acc(p, y_te, m)}
        for method, (onestep, rollout) in preds.items():
            for metric, fn in score.items():
                results[metric, regime, label, method] = {
                    split: (fn(onestep, gc == g), fn(rollout, gc == g))
                    for split, g in SPLITS}
        print(f"{label:13s} {regime:10s}  train {train.sum()}, test {test.sum()}")

GPT-4o-mini   subjective  train 320, test 320
Gemma-3n-E4B  subjective  train 320, test 320
Qwen3.5-9B    subjective  train 320, test 320
Llama-3-8B    subjective  train 320, test 320
GPT-4o-mini   objective   train 640, test 640
Gemma-3n-E4B  objective   train 640, test 640
Qwen3.5-9B    objective   train 640, test 640
Llama-3-8B    objective   train 640, test 640


In [19]:
# the tables: one-step (rollout) accuracy per cell, per metric
def cell(metric, regime, label, method, split):
    o, r = results[metric, regime, label, method][split]
    return f"{o:.1f} ({r:.1f})"

for metric in ("balanced", "raw"):
    for regime in REGIMES:
        print(f"=== {regime.capitalize()} questions — {METRIC_NAMES[metric]} ===")
        display(pd.DataFrame({m: {(l, sp): cell(metric, regime, l, m, sp)
                                  for _, l, _ in MODELS for sp, _ in SPLITS}
                              for m in METRIC_ROWS[metric]}).T)

=== Subjective questions — flip-and-class balanced accuracy ===


GPT-4o-mini              Gemma-3n-E4B               \
                                In-d.       Out-d.        In-d.       Out-d.   
Persistence               50.0 (64.5)  50.0 (64.0)  50.0 (59.3)  50.0 (59.3)   
Interaction-Free          50.6 (50.6)  48.6 (48.6)  60.8 (60.8)  59.6 (59.6)   
Mean-Field (Curie-Weiss)  57.9 (56.6)  59.3 (57.9)  68.3 (64.6)  67.9 (64.9)   
Discrete Update           77.7 (69.4)  75.7 (67.1)  62.4 (61.0)  61.3 (59.9)   
+ 3 Couplings             86.2 (76.4)  86.2 (77.1)  75.5 (68.0)  75.3 (66.8)   

                           Qwen3.5-9B                Llama-3-8B               
                                In-d.       Out-d.        In-d.       Out-d.  
Persistence               50.0 (65.1)  50.0 (65.1)  50.0 (63.2)  50.0 (61.7)  
Interaction-Free          49.1 (49.1)  47.5 (47.5)  47.7 (47.7)  45.6 (45.6)  
Mean-Field (Curie-Weiss)  65.2 (64.4)  65.7 (65.5)  54.8 (56.8)  53.5 (57.4)  
Discrete Update           64.6 (57.4)  60.5 (54.8)  54.0 (49.7)  51.7 (47.8)  
+ 3 Couplings             81.3 (70.8)  78.8 (69.2)  82.6 (68.4)  80.8 (68.3)

=== Objective questions — flip-and-class balanced accuracy ===


GPT-4o-mini              Gemma-3n-E4B               \
                                In-d.       Out-d.        In-d.       Out-d.   
Persistence               50.0 (55.6)  50.0 (55.0)  50.0 (61.8)  50.0 (62.2)   
Interaction-Free          49.8 (49.8)  49.3 (49.3)  59.0 (59.0)  59.1 (59.1)   
Mean-Field (Curie-Weiss)  65.4 (55.1)  65.3 (55.5)  72.7 (67.3)  72.7 (68.3)   
Discrete Update           70.1 (61.5)  67.8 (60.3)  59.7 (58.9)  59.7 (59.1)   
+ 3 Couplings             86.3 (68.2)  85.0 (65.9)  80.5 (68.5)  80.9 (68.5)   

                           Qwen3.5-9B                Llama-3-8B               
                                In-d.       Out-d.        In-d.       Out-d.  
Persistence               50.0 (53.9)  50.0 (53.2)  50.0 (57.0)  50.0 (56.8)  
Interaction-Free          44.6 (44.6)  45.4 (45.4)  50.0 (50.0)  50.0 (50.0)  
Mean-Field (Curie-Weiss)  73.6 (62.1)  73.4 (61.1)  64.7 (61.2)  64.5 (60.5)  
Discrete Update           50.6 (47.8)  49.8 (48.3)  50.9 (50.0)  50.6 (50.0)  
+ 3 Couplings             86.3 (65.7)  85.2 (64.2)  77.4 (62.9)  76.4 (60.8)

=== Subjective questions — raw accuracy ===


GPT-4o-mini              Gemma-3n-E4B               \
                                In-d.       Out-d.        In-d.       Out-d.   
Majority Class            64.0 (64.0)  61.9 (61.9)  54.0 (54.0)  55.9 (55.9)   
Persistence               76.9 (73.2)  73.7 (71.6)  81.3 (70.9)  81.1 (70.6)   
Interaction-Free          53.7 (53.7)  50.7 (50.7)  64.3 (64.3)  62.6 (62.6)   
Mean-Field (Curie-Weiss)  67.7 (60.9)  66.7 (62.1)  78.3 (73.5)  77.5 (73.9)   
Discrete Update           76.6 (69.2)  74.5 (67.3)  64.8 (64.1)  62.4 (62.0)   
+ 3 Couplings             86.3 (77.4)  85.4 (76.7)  81.1 (74.2)  79.9 (70.4)   

                           Qwen3.5-9B                Llama-3-8B               
                                In-d.       Out-d.        In-d.       Out-d.  
Majority Class            50.5 (50.5)  49.2 (49.2)  87.2 (87.2)  87.4 (87.4)  
Persistence               75.3 (75.2)  74.3 (74.8)  85.1 (75.3)  85.6 (74.4)  
Interaction-Free          48.4 (48.4)  46.3 (46.3)  49.6 (49.6)  48.9 (48.9)  
Mean-Field (Curie-Weiss)  73.6 (71.7)  74.0 (73.1)  86.4 (74.9)  86.2 (73.9)  
Discrete Update           63.5 (56.9)  58.3 (54.1)  55.8 (54.2)  54.0 (53.1)  
+ 3 Couplings             83.3 (75.1)  80.3 (71.8)  90.8 (81.2)  90.4 (81.2)

=== Objective questions — raw accuracy ===


GPT-4o-mini              Gemma-3n-E4B               \
                                In-d.       Out-d.        In-d.       Out-d.   
Majority Class            47.2 (47.2)  46.5 (46.5)  58.6 (58.6)  58.5 (58.5)   
Persistence               63.2 (58.6)  62.4 (57.7)  86.6 (78.6)  87.1 (79.3)   
Interaction-Free          48.9 (48.9)  47.9 (47.9)  61.6 (61.6)  62.0 (62.0)   
Mean-Field (Curie-Weiss)  69.5 (56.2)  69.3 (56.6)  90.2 (82.9)  90.4 (84.2)   
Discrete Update           68.8 (60.7)  66.2 (59.2)  61.2 (61.2)  61.6 (61.9)   
+ 3 Couplings             86.7 (67.4)  85.5 (64.9)  91.7 (81.8)  91.5 (81.6)   

                           Qwen3.5-9B                Llama-3-8B               
                                In-d.       Out-d.        In-d.       Out-d.  
Majority Class            49.1 (49.1)  49.4 (49.4)  79.4 (79.4)  77.9 (77.9)  
Persistence               81.8 (63.9)  82.4 (63.5)  75.7 (66.2)  75.8 (65.6)  
Interaction-Free          43.5 (43.5)  44.7 (44.7)  79.4 (79.4)  77.9 (77.9)  
Mean-Field (Curie-Weiss)  87.3 (69.6)  87.6 (68.6)  83.2 (75.0)  82.2 (72.4)  
Discrete Update           47.9 (46.3)  47.6 (47.1)  79.3 (79.0)  77.6 (77.3)  
+ 3 Couplings             91.3 (66.8)  90.9 (65.4)  89.3 (78.9)  88.4 (77.5)

In [20]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

# This cell has two modes: regenerate from in-memory results when available,
# or render the saved TeX export after a kernel restart.
def table_path(filename):
    local = Path(filename)
    repository = Path("res") / filename
    if local.exists():
        return local
    if repository.exists():
        return repository
    return repository if Path("res").is_dir() else local

latex_path = table_path("prediction_tables.tex")
has_results = all(name in globals() for name in (
    "results", "METRIC_ROWS", "METRIC_NAMES", "MODELS", "REGIMES", "SPLITS"))

if has_results:
    latex_method_names = {"+ 3 Couplings": r"\quad $+\,3$ Couplings"}
    latex_tables = []
    for metric in ("balanced", "raw"):
        rows_m = METRIC_ROWS[metric]
        lines = [r"\begin{table}[t]", r"\centering", r"\small",
                 r"\begin{adjustbox}{max width=\textwidth}",
                 r"\begin{tabular}{l cc cc cc cc}", r"\toprule",
                 " & " + " & ".join(
                     rf"\multicolumn{{2}}{{c}}{{{label}}}"
                     for _, label, _ in MODELS) + r" \\",
                 "".join(rf"\cmidrule(lr){{{2 * i}-{2 * i + 1}}}"
                         for i in range(1, 5)),
                 "Method & " + " & ".join(["In-d. & Out-d."] * 4) + r" \\"]
        for regime in REGIMES:
            lines += [r"\midrule",
                      rf"\multicolumn{{9}}{{l}}{{\textit{{{regime.capitalize()} Questions}}}} \\",
                      r"\midrule"]
            for method in rows_m:
                cells = []
                for _, label, _ in MODELS:
                    for split, _ in SPLITS:
                        one_step, rollout = results[
                            metric, regime, label, method][split]
                        best = max(results[
                            metric, regime, label, candidate][split][0]
                                   for candidate in rows_m)
                        value = f"{one_step:.1f} ({rollout:.1f})"
                        cells.append(
                            rf"\textbf{{{value}}}" if one_step == best else value)
                method_tex = latex_method_names.get(method, method)
                lines.append(method_tex + " & " + " & ".join(cells) + r" \\")
        lines += [r"\bottomrule", r"\end{tabular}", r"\end{adjustbox}",
                  rf"\caption{{\textit{{Prediction.}} "
                  rf"{METRIC_NAMES[metric].capitalize()} "
                  r"of predictions on held-out test questions.}",
                  r"\end{table}"]
        latex_tables.append("\n".join(lines))
    latex_fragment = "\n\n".join(latex_tables)
    standalone_latex = "\n".join([
        r"\documentclass{article}",
        r"\usepackage{booktabs}",
        r"\usepackage{adjustbox}",
        r"\begin{document}",
        latex_fragment,
        r"\end{document}",
    ])
    latex_path.write_text(standalone_latex + "\n", encoding="utf-8")
    status = f"Regenerated {latex_path} from in-memory results."
else:
    if not latex_path.exists():
        raise FileNotFoundError(
            f"Saved table file not found: {latex_path}.")
    standalone_latex = latex_path.read_text(encoding="utf-8")
    latex_tables = re.findall(
        r"\\begin\{table\}.*?\\end\{table\}",
        standalone_latex, flags=re.DOTALL)
    status = f"Rendered saved results from {latex_path}; no models were run."

if len(latex_tables) != 2:
    raise ValueError(f"Expected two tables in {latex_path}, found {len(latex_tables)}.")

preview_columns = pd.MultiIndex.from_product([
    ("GPT-4o-mini", "Gemma-3n-E4B", "Qwen3.5-9B", "Llama-3-8B"),
    ("In-d.", "Out-d."),
])

def clean_latex(text):
    text = re.sub(r"\\textbf\{([^{}]+)\}", r"\1", text)
    text = text.replace(r"\quad", "").replace(r"\,", "")
    text = text.replace("$", "").removesuffix(r"\\").strip()
    return text.replace("+3 Couplings", "+ 3 Couplings")

def preview_sections(table_text):
    sections = {"Subjective": [], "Objective": []}
    current = None
    for line in table_text.splitlines():
        if "Subjective Questions" in line:
            current = "Subjective"
            continue
        if "Objective Questions" in line:
            current = "Objective"
            continue
        if current is None or "&" not in line or not line.rstrip().endswith(r"\\"):
            continue
        cells = [clean_latex(cell) for cell in line.split("&")]
        if len(cells) == 9 and cells[0] != "Method":
            sections[current].append(cells)
    return sections

for metric_name, table_text in zip(
        ("Flip-and-class balanced accuracy", "Raw accuracy"), latex_tables):
    for regime, rows in preview_sections(table_text).items():
        frame = pd.DataFrame(
            [row[1:] for row in rows],
            index=[row[0] for row in rows],
            columns=preview_columns)
        display(frame.style.set_caption(f"{regime} questions — {metric_name}"))

print(status)


Regenerated prediction_tables.tex from in-memory results.


In [21]:
# fitted couplings (R1.1b): the discrete update refit at one / three / five
# couplings; three and five are collinear, so beta_0 (the unsigned
# neighborhood (|J| s)) is fit first and the signed betas second
BETA_LABELS = {"one": ["beta"], "three": ["beta_pos", "beta_neg", "beta_0"],
               "five": ["beta_pos_T", "beta_pos_F", "beta_neg_T",
                        "beta_neg_F", "beta_0"]}

def truth_drives(J_e, s, tau):
    """Paper convention (Section 5.4): each signed channel splits by whether
    the NEIGHBOR currently holds the correct answer, so beta_T multiplies the
    drive from correct neighbors on both channels — column order
    [J+ s_T, J+ s_F, J- s_T, J- s_F] to match BETA_LABELS['five']."""
    Jp, Jn = np.maximum(J_e, 0), np.minimum(J_e, 0)
    sT = np.where(s * tau[:, None, None] > 0, s, 0.0)  # neighbors on correct side
    sF = np.where(s * tau[:, None, None] < 0, s, 0.0)
    return np.stack([np.einsum("eij,etj->eti", Jp, sT),
                     np.einsum("eij,etj->eti", Jp, sF),
                     np.einsum("eij,etj->eti", Jn, sT),
                     np.einsum("eij,etj->eti", Jn, sF)], axis=-1)

couplings = {}
for model, label, short in MODELS:
    couplings[short] = {}
    for regime in REGIMES:
        x, y, ep = build_xy(model, regime, runs, banks, P, GCLASS)
        train = (ep["split"] == "train") & np.isin(ep["gcls"], ["seen", "train_only"])
        rows = x[train].reshape(-1, x.shape[-1])
        parsed = y[train].reshape(-1) != 0
        y01 = (y[train].reshape(-1)[parsed] > 0).astype(float)
        pos, neg = rows[:, D_POS], rows[:, D_NEG]
        clf = Logistic().fit(np.concatenate(
            [rows[:, FIELD], (pos + neg)[:, None]], axis=-1)[parsed], y01)
        # five splits each signed drive by whether the neighbor holds the
        # correct answer (objective only); three and five share the two-stage
        # recipe
        signed = {"three": np.stack([pos, neg], axis=-1),
                  "five": truth_drives(ep["J"][train], x[train][..., S_PREV],
                                       ep["tau"][train]).reshape(-1, 4)}
        cc = couplings[short][regime] = {
            "one": {"betas": {"beta": float(clf.w[16])}, "field": clf.w[:16].tolist()}}
        for tier in ("three", "five"):
            if tier == "five" and regime == "subjective":
                cc[tier] = None                      # needs a ground truth
                continue
            w = two_stage_fit(rows[:, FIELD], signed[tier], pos - neg, parsed, y01)
            cc[tier] = {"betas": dict(zip(BETA_LABELS[tier], w[16:].tolist())),
                        "field": w[:16].tolist()}

BETA_TEX = {"beta": "β", "beta_pos": "β⁺", "beta_neg": "β⁻", "beta_0": "β₀",
            "beta_pos_T": "β⁺T", "beta_pos_F": "β⁺F",
            "beta_neg_T": "β⁻T", "beta_neg_F": "β⁻F"}
pd.DataFrame({(label, rg): {(tier.capitalize(), BETA_TEX[lab]):
                            (round(couplings[short][rg][tier]["betas"][lab], 2)
                             if couplings[short][rg][tier] else None)
                            for tier in ("one", "three", "five")
                            for lab in BETA_LABELS[tier]}
              for _, label, short in MODELS for rg in REGIMES}).T

One Three              Five                        
                            β    β⁺    β⁻    β₀   β⁺T   β⁺F   β⁻T   β⁻F    β₀
GPT-4o-mini  subjective  1.00  2.38  1.08  0.61   NaN   NaN   NaN   NaN   NaN
             objective   0.63  2.10  0.75  0.93  2.29  1.98  0.70  0.85  0.93
Gemma-3n-E4B subjective  0.30  0.44  0.35  0.55   NaN   NaN   NaN   NaN   NaN
             objective   0.06  0.29  0.23  0.96  0.34  0.20  0.17  0.36  0.96
Qwen3.5-9B   subjective  0.57  1.12  0.65  0.64   NaN   NaN   NaN   NaN   NaN
             objective   0.29  0.90  0.50  1.01  1.03  0.77  0.40  0.80  1.01
Llama-3-8B   subjective  0.51  1.35  0.67  0.99   NaN   NaN   NaN   NaN   NaN
             objective   0.44  0.97  0.61  0.97  1.15  0.84  0.59  0.61  0.97

In [22]:
# save the couplings (+ each tier's 16 field weights) for the downstream
# notebooks (5_temperaturesweep, 6_distribution)
couplings_path = RES_DIR / "couplings.json"
couplings_path.write_text(json.dumps({
    "rule": "full-batch Adam logistic fits (standardized columns, small "
            "l2 = 1e-3 on the field weights; bias and couplings unpenalized, "
            "couplings initialized at 0.1) of the discrete update on the "
            "train-question episodes over the 8 signed random graphs; designs "
            "one = [field | (J s)], three = [field | (J+ s) | (J- s) | (|J| s)], "
            "five = each signed channel split by whether the neighbor holds "
            "the correct answer (paper Section 5.4 convention: beta_T "
            "multiplies the drive from correct neighbors on both channels; "
            "objective only). Collinear tiers are fit in two stages: beta_0 on "
            "the unsigned neighborhood first, then the signed betas against "
            "the fixed beta_0 offset; effective signed couplings are "
            "beta_pos + beta_0 and beta_neg - beta_0",
    "models": {sh: l for _, l, sh in MODELS},
    "regimes": REGIMES,
    "beta_labels": BETA_LABELS,
    "params": couplings}, indent=1), encoding="utf-8")
print(f"wrote {couplings_path}")

wrote couplings.json
